# TML — Paperspace Free M4000 RapidOCR benchmark
This notebook benchmarks the free Paperspace Gradient GPU on the same Gallica newspaper workload used by TML. It tests 1, 2, 3 and 4 CUDA worker processes with the HQ RapidOCR profile.

Expected free machine: **Free GPU (NVIDIA M4000)**. Run all cells.


In [ ]:
import subprocess, sys, os, pathlib
subprocess.run(['nvidia-smi'], check=True)
REPO='/notebooks/tml-paperspace-bench-repo'
if os.path.isdir(REPO):
    subprocess.run(['git','-C',REPO,'pull','--ff-only'], check=True)
else:
    subprocess.run(['git','clone','-q','https://github.com/Tennismylife/Tennis-OCR-Pipeline.git',REPO], check=True)
print('REPO_READY', REPO, flush=True)


In [ ]:
import subprocess, sys
subprocess.run([sys.executable,'-m','pip','uninstall','-y','onnxruntime','onnxruntime-gpu'], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
subprocess.run([sys.executable,'-m','pip','install','-q','-r',f'{REPO}/colab/requirements.txt'], check=True)
try:
    import torch
    print('TORCH', torch.__version__, 'CUDA', torch.version.cuda, 'AVAILABLE', torch.cuda.is_available(), flush=True)
except Exception as e:
    print('TORCH_PRELOAD_WARNING', type(e).__name__, e, flush=True)
import onnxruntime as ort
try:
    if hasattr(ort,'preload_dlls'): ort.preload_dlls()
except Exception as e:
    print('ORT_PRELOAD_WARNING', type(e).__name__, e, flush=True)
providers=ort.get_available_providers()
print('ORT', ort.__version__, 'PROVIDERS', providers, flush=True)
if 'CUDAExecutionProvider' not in providers:
    raise RuntimeError('CUDAExecutionProvider unavailable on this Paperspace runtime')
print('CUDA_ENV_READY', flush=True)


In [ ]:
import subprocess, sys
cmd=[sys.executable,'-u',f'{REPO}/paperspace/m4000_benchmark.py','--sample-pages','18','--candidates','1,2,3,4','--profile','HQ']
print('START', ' '.join(cmd), flush=True)
subprocess.run(cmd, check=True)


In [ ]:
import json, pathlib
p=pathlib.Path('/notebooks/tml_m4000_benchmark/result.json')
r=json.loads(p.read_text())
print(json.dumps(r, indent=2))
if r.get('best'):
    b=r['best']
    print(f"BEST: {b['workers']} workers — {b['pages_per_min']} pages/min")
